In [23]:
######Match_Soldiers_To_Parents.py
#6/27/2025


from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance

from dateutil import parser
from dateutil.relativedelta import relativedelta

def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def  display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        input('')

    
def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ','PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ',\
                'Tec4','PVT',  'Capt', 'CPL', 'III',  'II',  '2Lt','SSgt', '1Lt','Tec5','TSgt','Sgt',  'CAPT', 'SK2']
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df     

def discard_empty_sets(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:x.discard('') or x if isinstance(x,set) else x)
    return dataframe_df

def  trim_dataframe_set(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:{item.strip() for item in x if isinstance(item,str)} if isinstance(x,set) else x)
    return dataframe_df


maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Match_Soldiers_To_Parents'
print('**Starting***',process_name,'***,Time:',start_time,'**\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

grave_source='ABMC'
letter='A'
#grave_source='Arlington'

#relative_list=['Parents','Siblings','Spouses','Children']

fileInputSoldier=grave_source+"_Relative_Parents_Table_Out"
fileInputABMC=grave_source+"_Relative_Pre_"+letter
fileInputCemetery="Categorized_Cemetery_List_Preliminary"
fileOutput=grave_source+"_Soldier_Parent_Merge_"+letter

fileInSold=DirMain+"Analysis\\"+fileInputSoldier+".csv"
fileInABMC=DirMain+"Analysis\\"+fileInputABMC+".csv"
fileInCem=DirMain+"OutputData\\Cemeteries\\"+fileInputCemetery+".csv"
fileOut=DirMain+"Analysis\\"+fileOutput+".csv"

soldiers_df=pd.read_csv(fileInSold,encoding='ISO-8859-1')
abmc_fag_df=pd.read_csv(fileInABMC,encoding='ISO-8859-1',low_memory=False)
cemetery_df=pd.read_csv(fileInCem,encoding='ISO-8859-1')

soldiers_df=soldiers_df[soldiers_df['name_index'].str[0]==letter]

field_list_soldiers=['name_index','last_name','first_name','middle_name','soldiers_full_name','birth_date','birth_index',\
'death_date','death_index','entry_state','birth_city','birth_state','Parents_name_index','Parents_name_full']

field_list_fag=['name_index','name_index_short','last_name','first_name','middle_name','full_name',\
'birth_date','death_date','birth_index','death_index','birth_city','birth_state','birth_country',\
'cemetery_name','cemetery_id','url','spouses','children']

soldiers_df['birth_index']=soldiers_df['birth_index'].fillna(100001010).astype(int)
soldiers_df['death_index']=soldiers_df['death_index'].fillna(100001010).astype(int)

#soldiers_df['birth_date']=pd.to_datetime(soldiers_df['birth_date'].fillna('01/01/1000'),format='%m/%d/%Y',errors='coerce')
#soldiers_df['death_date']=pd.to_datetime(soldiers_df['death_date'].fillna('01/01/1000'),format='%m/%d/%Y',errors='coerce')

abmc_fag_df['birth_index']=abmc_fag_df['birth_index'].fillna(100001010).astype(int)
abmc_fag_df['death_index']=abmc_fag_df['death_index'].fillna(100001010).astype(int)
abmc_fag_df['birth_date']=pd.to_datetime(abmc_fag_df['birth_date'].fillna('01/01/2000'),format='mixed')
#abmc_fag_df['death_date']=pd.to_datetime(abmc_fag_df['death_date'].fillna('01/01/1000'),format='%m/%d/%Y')

#CleanUpsoldiers_dfAndabmc_fag_df
column_s='Parents_name_full'
soldiers_df=clean_dataframe(soldiers_df,column_s)
column_s='soldiers_full_name'
soldiers_df=clean_dataframe(soldiers_df,column_s)

abmc_fag_df_column_list=['spouses','children','full_name']
for column_fag in abmc_fag_df_column_list:
    abmc_fag_df=clean_dataframe(abmc_fag_df,column_fag)

soldiers_df_column_list=['Parents_name_full','soldiers_full_name']
for column in soldiers_df_column_list:
    column_set=column+'_set'
    soldiers_df_split=soldiers_df[column].str.split(" ")
    soldiers_df[column_set]=soldiers_df_split.apply(set)
    soldiers_df=discard_empty_sets(soldiers_df,column_set)
    soldiers_df=trim_dataframe_set(soldiers_df,column_set)
# print(soldiers_df['soldiers_full_name_set'])    
# input('')


fag_df_column_list=['spouses','full_name']
for column in fag_df_column_list:
    column_set=column+'_set'
    abmc_fag_df_split=abmc_fag_df[column].str.split(" ")
    abmc_fag_df[column_set]=abmc_fag_df_split.apply(set)
    abmc_fag_df=discard_empty_sets(abmc_fag_df,column_set)
    abmc_fag_df=trim_dataframe_set(abmc_fag_df,column_set)   
column='children'
column_set=column+'_set'
abmc_fag_df_split=abmc_fag_df[column].str.split("*")
abmc_fag_df[column_set]=abmc_fag_df_split.apply(set)
abmc_fag_df=discard_empty_sets(abmc_fag_df,column_set)
abmc_fag_df=trim_dataframe_set(abmc_fag_df,column_set)
column_set_2=column_set+'_of_sets'
abmc_fag_df[column_set_2]=abmc_fag_df[column_set].apply(
lambda x:{frozenset(item.strip().split())for item in x if isinstance(item,str)and item.strip()} if isinstance(x,set) else x)
#*********forindividualchildrensets---set(list(abmc_fag_df[column_set_2].iloc[35])[0])

soldier_abmc_df=pd.merge(
soldiers_df,
abmc_fag_df,
left_on='Parents_name_index',
right_on='name_index_short',
how='inner'#or'left','inner','right','outer'dependingonwhatyouwant
)
#*****************Computeageatbirth*********************
soldier_abmc_df['birth_date_y']=pd.to_datetime(soldier_abmc_df['birth_date_y'],errors='coerce')
soldier_abmc_df['birth_date_x']=pd.to_datetime(soldier_abmc_df['birth_date_x'],errors='coerce')
soldier_abmc_df['age_at_birth']=((soldier_abmc_df['birth_date_x']-soldier_abmc_df['birth_date_y']).dt.days//365.25)
soldier_abmc_df['age_at_birth']=soldier_abmc_df['age_at_birth'].fillna(0).astype(int)

soldier_abmc_df['soldier_abmc_missing']=soldier_abmc_df['Parents_name_full_set']-soldier_abmc_df['full_name_set']
soldier_abmc_match_df=soldier_abmc_df[soldier_abmc_df['soldier_abmc_missing'].apply(lambda x:len(x)==0)]
soldier_abmc_match_df.reset_index(drop=True,inplace=True)

soldier_abmc_category_df=pd.merge(
    soldier_abmc_match_df,
    cemetery_df,
    left_on='cemetery_id',
    right_on='cemetery_id',
    how='left'#or'left','inner','right','outer'dependingonwhatyouwant
).sort_values(['name_index', 'full_name','spouses'])#.drop_duplicates
soldier_abmc_category_df.drop_duplicates


field_list_soldier_abmc_reordered=['name_index','last_name_x','first_name_x','middle_name_x','soldiers_full_name','death_index_x','entry_state',\
                'birth_state_x','Parents_name_index','Parents_name_full','Parents_name_full_set','soldiers_full_name_set','age_at_birth','soldier_abmc_missing',\
                'name_index_short','full_name','full_name_set','spouses_set','birth_state_y','birth_country',\
                'category','cemetery_name_x','cemetery_id','url','spouses','children','children_set','children_set_of_sets']

field_list_soldier_abmc_rename=['soldier_name_index','soldier_last_name','soldier_first_name','soldier_middle_name','soldiers_full_name','soldier_death_index','entry_state',\
                'soldier_birth_state','parents_name_index','parents_name_full','parents_name_full_set','soldiers_full_name_set','age_at_birth','soldier_abmc_missing',\
                'match_name_index_short','match_full_name','match_full_name_set','spouses_set','parents_birth_state','parents_birth_country',\
                'category','cemetery_name','cemetery_id','url','spouses','children','children_set','children_set_of_sets']

soldier_parent_match_df=soldier_abmc_category_df[field_list_soldier_abmc_reordered]
soldier_parent_match_df.columns=field_list_soldier_abmc_rename
soldier_parent_match_df.to_csv(fileOut,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOut)

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** Completed **  {process_name}   Processing Time: {processing_time_seconds} Sec")   




**Starting*** Match_Soldiers_To_Parents ***,Time: 2025-07-01 20:34:18.979701 **



UploadedTo: C:\Users\jrozm\OneDrive\Benjamin\Analysis\ABMC_Soldier_Parent_Merge_A.csv
**** Completed **  Match_Soldiers_To_Parents   Processing Time: 98 Sec


In [18]:
list(soldier_abmc_df)

['name_index',
 'last_name_x',
 'first_name_x',
 'middle_name_x',
 'soldiers_full_name',
 'birth_date_x',
 'birth_index_x',
 'death_date_x',
 'death_index_x',
 'entry_state',
 'birth_city_x',
 'birth_state_x',
 'Parents_name_index',
 'Parents_name_full',
 'Parents_name_full_set',
 'soldiers_full_name_set',
 'ï»¿name_index',
 'name_index_short',
 'last_name_y',
 'first_name_y',
 'middle_name_y',
 'full_name',
 'birth_date_y',
 'death_date_y',
 'birth_index_y',
 'death_index_y',
 'birth_city_y',
 'birth_state_y',
 'birth_country',
 'cemetery_name',
 'cemetery_id',
 'url',
 'spouses',
 'children',
 'spouses_set',
 'full_name_set',
 'children_set',
 'children_set_of_sets',
 'age_at_birth',
 'soldier_abmc_missing']

In [ ]:
soldier_abmc_category_df=pd.merge(
    soldier_abmc_match_df,
    cemetery_df,
    left_on='cemetery_id',
    right_on='cemetery_id',
    how='left'#or'left','inner','right','outer'dependingonwhatyouwant
).sort_values(['name_index', 'full_name','spouses'])#.drop_duplicates
soldier_abmc_category_df.drop_duplicates

field_list_soldier_abmc_reordered=['name_index','last_name_x','first_name_x','middle_name_x','death_index_x','entry_state',\
                'birth_state_x','Parents_name_index','Parents_name_full','Parents_name_full_set','age_at_birth','soldier_abmc_missing',\
                'name_index_short','full_name','full_name_set','spouses_set','birth_state_y','birth_country',\
                'category','cemetery_name_x','cemetery_id','url','spouses','children','children_set','children_set_of_sets']

field_list_soldier_abmc_rename=['soldier_name_index','soldier_last_name','soldier_first_name','soldier_middle_name','soldier_death_index','entry_state',\
                'soldier_birth_state','parents_name_index','parents_name_full','parents_name_full_set','age_at_birth','soldier_abmc_missing',\
                'match_name_index_short','match_full_name','match_full_name_set','spouses_set','parents_birth_state','parents_birth_country',\
                'category','cemetery_name','cemetery_id','url','spouses','children','children_set','children_set_of_sets']

soldier_parent_match_df=soldier_abmc_category_df[field_list_soldier_abmc_reordered]
soldier_parent_match_df.columns=field_list_soldier_abmc_rename
soldier_parent_match_df.to_csv(fileOut,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOut)

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** Completed **  {process_name}   Processing Time: {processing_time_seconds} Sec")   

In [9]:
soldiers_df=soldiers_df[soldiers_df['name_index'].str[0]==letter]
soldiers_df

,name_index,last_name,first_name,middle_name,birth_date,birth_index,death_date,death_index,entry_state,birth_city,birth_state,Parents_name_index,Parents_name_full,Parents_name_full_set
0,Aaron-George-19441217,Aaron,George,Edward,1922-01-26,19220126,1944-12-17,19441217,Pennsylvania,Brackenridge,Pennsylvania,Aaron-James,James Burke Aaron,"{Burke, James, Aaron}"
1,Aaron-George-19441217,Aaron,George,Edward,1922-01-26,19220126,1944-12-17,19441217,Pennsylvania,Brackenridge,Pennsylvania,Aaron-Sarah,Sarah T Hollobaugh Aaron,"{Aaron, Hollobaugh, Sarah, T}"
2,Aaron-John-19441127,Aaron,John,Daniel,1923-02-10,19230210,1944-11-27,19441127,Oklahoma,Chismville,Arkansas,Aaron-Margaret,Margaret Adaline Addie Lee Aaron,"{Addie, Margaret, Aaron, Lee, Adaline}"
3,Aaron-John-19441127,Aaron,John,Daniel,1923-02-10,19230210,1944-11-27,19441127,Oklahoma,Chismville,Arkansas,Aaron-Russell,Russell Preston Aaron,"{Aaron, Russell, Preston}"
4,Abbott-Alexander-19450120,Abbott,Alexander,Winn,1919-03-06,19190306,1945-01-20,19450120,Connecticut,Cambridge,Massachusetts,Abbott-Alexander,Alexander Hewes Abbott,"{Hewes, Alexander, Abbott}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1153,Austin-Richard-19450114,Austin,Richard,Gilbert,1916-12-03,19161203,1945-01-14,19450114,Pennsylvania,Tyrone,Pennsylvania,Austin-James,James Blaine Austin,"{James, Austin, Blaine}"
1154,Austin-Robert-19431014,Austin,Robert,Leroy,1920-07-12,19200712,1943-10-14,19431014,Maryland,Ryceville,Maryland,Austin-Carrie,Carrie D Ryce Austin,"{D, Carrie, Austin, Ryce}"
1155,Austin-Robert-19431014,Austin,Robert,Leroy,1920-07-12,19200712,1943-10-14,19431014,Maryland,Ryceville,Maryland,Austin-Everett,Everett Leroy Austin,"{Leroy, Austin, Everett}"
1156,Austin-Willard-19441229,Austin,Willard,Melvin,1923-05-17,19230517,1944-12-29,19441229,New York,Gloversville,New York,Austin-Ethel,Ethel Almedia Williams Austin,"{Williams, Almedia, Austin, Ethel}"


In [93]:
abmc_fag_df=pd.read_csv(fileInABMC,encoding='ISO-8859-1',low_memory=False)

In [99]:
list(soldier_parent_match_df)

['soldier_name_index',
 'soldier_last_name',
 'soldier_first_name',
 'soldier_middle_name',
 'soldier_death_index',
 'entry_state',
 'soldier_birth_state',
 'parents_name_index',
 'parents_name_full',
 'parents_name_full_set',
 'age_at_birth',
 'soldier_abmc_missing',
 'match_name_index_short',
 'match_full_name',
 'match_full_name_set',
 'spouses_set',
 'parents_birth_state',
 'parents_birth_country',
 'category',
 'cemetery_name',
 'cemetery_id',
 'url',
 'spouses',
 'children',
 'children_set',
 'children_set_of_sets']